In [1]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  !pip install tf-nightly
except Exception:
  pass
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 654.2/654.2 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 64.8 MB/s eta 0:00:00
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.1 which is incompatible.
2.20.0-dev20250311


In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

--2025-03-14 13:47:37--  https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 172.67.70.149, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 358233 (350K) [text/tab-separated-values]
Saving to: ‘train-data.tsv’

train-data.tsv      100%[===================>] 349.84K  --.-KB/s    in 0.04s   

2025-03-14 13:47:37 (9.41 MB/s) - ‘train-data.tsv’ saved [358233/358233]

--2025-03-14 13:47:37--  https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 172.67.70.149, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 118774 (116K) [text/tab-separated-values]
Saving to: ‘valid-data.tsv’

valid-data.tsv      100%[==============

In [3]:
dftrain = pd.read_csv(train_file_path, sep="\t", header=0, names=['label', 'text'], dtype={'label': 'str', 'text': 'str'})
dftest = pd.read_csv(test_file_path, sep="\t", header=0, names=['label', 'text'], dtype={'label': 'str', 'text': 'str'})

#Replace label ['ham', 'spam'] with [0,1]
vocab_label = np.unique(dftrain['label'])
dftrain['label'] = dftrain['label'].replace(vocab_label, range(len(vocab_label)))
dftest['label'] = dftest['label'].replace(vocab_label, range(len(vocab_label)))

<ipython-input-3-4700bc3f5c43>:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dftrain['label'] = dftrain['label'].replace(vocab_label, range(len(vocab_label)))
<ipython-input-3-4700bc3f5c43>:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dftest['label'] = dftest['label'].replace(vocab_label, range(len(vocab_label)))


In [4]:
#Create the text encoder

#VOCAB_SIZE = 7700
#MAX_WORDS = dftrain['text'].str.split().str.len().max()
#encoder = tf.keras.layers.TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAX_WORDS)

encoder = tf.keras.layers.TextVectorization()
encoder.adapt(dftrain['text'])

In [5]:
vocab = np.array(encoder.get_vocabulary())
vocab[:20]
#encoder.vocabulary_size()

array(['', '[UNK]', 'to', 'i', 'you', 'a', 'the', 'u', 'and', 'in', 'is',
       'me', 'my', 'for', 'your', 'of', 'it', 'call', 'have', 'on'],
      dtype='<U48')

In [6]:
#Convert pandas dataframe to Tensor
train_dataset = tf.data.Dataset.from_tensor_slices((dftrain['text'], dftrain['label']))
test_dataset = tf.data.Dataset.from_tensor_slices((dftest['text'], dftest['label']))

BUFFER_SIZE = 10000
BATCH_SIZE = 32

train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [7]:
for example, label in train_dataset.take(1):
  print('texts: ', example.numpy()[:3])
  print()
  print('labels: ', label.numpy()[:3])

texts:  [b"i dun thk i'll quit yet... hmmm, can go jazz ? yogasana oso can... we can go meet em after our lessons den..."
 b"sorry, i'll call later"
 b"ee msg na poortiyagi odalebeku: hanumanji 7 name 1-hanuman 2-bajarangabali 3-maruti 4-pavanaputra 5-sankatmochan 6-ramaduth 7-mahaveer ee 7 name  &lt;#&gt;  janarige ivatte kalisidare next saturday olage ondu good news keluviri...! maretare inde 1 dodda problum nalli siguviri idu matra  &lt;#&gt; % true.. don't neglet."]

labels:  [0 0 0]


In [8]:
#Example
encoded_example = encoder(example)[:3].numpy()
encoded_example

array([[   3,  242,  216,   62, 2012,  256, 1101,   31,   44, 1696, 1862,
         477,   31,   39,   31,   44,  180,  888,  151,   84, 1089,  357,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0],
       [  90,   62,   17,   99,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0],
       [3166,  160, 1663, 5052, 5310, 6254,  434,  224, 7960, 7921, 7867,
        7822, 7799, 7772, 7757, 3166,  434,  224,   48, 5958, 5970, 5897,
         201,  622, 5287, 5279,   61,  747, 5877, 5630, 6048,  159, 6792,
        4967, 5443, 4555, 6085, 5614,   48,  489,   47, 5415]])

In [9]:
for n in range(3):
  print("Original: ", example[n].numpy())
  print("Round-trip: ", " ".join(vocab[encoded_example[n]]))
  print()

Original:  b"i dun thk i'll quit yet... hmmm, can go jazz ? yogasana oso can... we can go meet em after our lessons den..."
Round-trip:  i dun thk ill quit yet hmmm can go jazz yogasana oso can we can go meet em after our lessons den                    

Original:  b"sorry, i'll call later"
Round-trip:  sorry ill call later                                      

Original:  b"ee msg na poortiyagi odalebeku: hanumanji 7 name 1-hanuman 2-bajarangabali 3-maruti 4-pavanaputra 5-sankatmochan 6-ramaduth 7-mahaveer ee 7 name  &lt;#&gt;  janarige ivatte kalisidare next saturday olage ondu good news keluviri...! maretare inde 1 dodda problum nalli siguviri idu matra  &lt;#&gt; % true.. don't neglet."
Round-trip:  ee msg na poortiyagi odalebeku hanumanji 7 name 1hanuman 2bajarangabali 3maruti 4pavanaputra 5sankatmochan 6ramaduth 7mahaveer ee 7 name ltgt janarige ivatte kalisidare next saturday olage ondu good news keluviri maretare inde 1 dodda problum nalli siguviri idu matra ltgt true dont negl

In [10]:
model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(
        input_dim=len(encoder.get_vocabulary()),
        output_dim=64,
        # Use masking to handle the variable sequence lengths
        mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [11]:
#All the layers after the Embedding support masking.
print([layer.supports_masking for layer in model.layers])

[False, True, True, True, True]


In [12]:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(),
              optimizer=tf.keras.optimizers.Adam(1e-4),
              metrics=['accuracy'])

In [13]:
history = model.fit(train_dataset, epochs=10,
                    validation_data=test_dataset,
                    validation_steps=30)

Epoch 1/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 17s 81ms/step - accuracy: 0.8325 - loss: 0.6577 - val_accuracy: 0.8604 - val_loss: 0.4532
Epoch 2/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 20s 157ms/step - accuracy: 0.8802 - loss: 0.3612 - val_accuracy: 0.9531 - val_loss: 0.1909
Epoch 3/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 11s 83ms/step - accuracy: 0.9789 - loss: 0.1390 - val_accuracy: 0.9833 - val_loss: 0.0848
Epoch 4/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 11s 85ms/step - accuracy: 0.9873 - loss: 0.0721 - val_accuracy: 0.9844 - val_loss: 0.0622
Epoch 5/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 20s 84ms/step - accuracy: 0.9927 - loss: 0.0486 - val_accuracy: 0.9854 - val_loss: 0.0567
Epoch 6/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 20s 157ms/step - accuracy: 0.9974 - loss: 0.0259 - val_accuracy: 0.9875 - val_loss: 0.0529
Epoch 7/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 41s 158ms/step - accuracy: 0.9990 - loss: 0.0159 - val_accuracy: 0.9875 - val_loss: 0.0474
Epoch 8/10
131/131 ━━━━━━━━━━━━━━━━━━━━ 31s 82ms/step - accuracy: 0.9990 - loss: 0.0148

In [14]:
sample_text = ('The movie was cool. The animation and the graphics '
               'were out of this world. I would recommend this movie.')
predictions = model(np.array([sample_text]))
predictions[0]

<tf.Tensor: shape=(1,), dtype=float32, numpy=array([0.06410247], dtype=float32)>

In [19]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
  predictions = model(np.array([pred_text]))
  prediction = list(predictions[0].numpy())
  if prediction[0] <= 0.5:
    prediction.append(vocab_label[0])
  else:
    prediction.append(vocab_label[1])
  return (prediction)

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

[0.0013461693, 'ham']


In [20]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()


You passed the challenge. Great job!
